# Finding optimal GPG pulses for recovery

This notebook starts from the `(b, g, m) = (3, 3, 1)` permutation-invariant code, evolves the code state through exact collective amplitude damping, constructs the four-branch Petz/Barnum-Knill recovery used by the GPG recovery circuit, and shows how to optimize detuned GPG pulse sequences for the state-preparation targets that appear in that recovery.

The first optimization below prepares one arbitrary Dicke-space state. The later cells use the same state-preparation optimizer on recovery targets extracted from the exact split/feedback decomposition.

In [ ]:
import numpy as np
import pandas as pd
import qutip as qt

from qer import gpgs
from qer.bk_recovery import petz_recovery_kraus
from qer.codewords import bgmcode_kets_in_top_block
from qer.noisemodel import noisemodel

## Code state and exact collective AD

The GPG routines work in the symmetric Dicke top block of dimension `N + 1`. The exact collective amplitude-damping channel is generated in the PIQS reduced space; `gpgs.apply_channel_to_state` embeds the top block, applies the exact channel, and projects back to the top block.

In [ ]:
b, g, m = 3, 3, 1
gamma = 1e-3
dt = 1.0
cooperativity = 1e8

ket0, ket1, num_qubits = bgmcode_kets_in_top_block(b, g, m, return_qutip=True)
system_dim = num_qubits + 1
rho_code = (ket0 * ket0.dag() + ket1 * ket1.dag()) / 2

exact_ad = noisemodel(
    "global symmetric amplitude damping",
    num_qubits,
    gamma,
    dt,
    return_rep="super",
    dynamics="exact",
)
rho_noisy = gpgs.apply_channel_to_state(exact_ad, rho_code)

print(f"N = {num_qubits}, Dicke top-block dimension = {system_dim}")
print(f"Trace after exact collective AD = {np.real(rho_noisy.tr()):.12f}")

## Four-branch recovery map

The three-ancilla GPG recovery construction expects four recovery Kraus operators. Here the physical state is still evolved with exact collective AD, while the four recovery branches are built from the approximate collective-AD Kraus model and the code projector. This is the same split used by the pulse-search workflow in `scripts/parallel_noisy_gpg_search.py`.

In [ ]:
approx_ad_kraus = noisemodel(
    "global symmetric amplitude damping",
    num_qubits,
    gamma,
    dt,
    return_rep="kraus",
    dynamics="approx",
)
approx_ad_top = gpgs.restrict_operators_to_dimension(approx_ad_kraus, system_dim)
recovery_ops = petz_recovery_kraus(approx_ad_top, rho_code)

rho_direct = gpgs.direct_recovery_state(recovery_ops, rho_noisy)
direct_fidelity = gpgs.output_fidelity(
    rho_direct,
    rho_code,
    logical_kets=(ket0, ket1),
)

print(f"Recovery branches = {len(recovery_ops)}")
print(f"Direct Petz/BK recovered codespace fidelity = {direct_fidelity:.12f}")

## Split the recovery into GPG synthesis targets

`build_lv_recovery_data` polar-decomposes each recovery branch, constructs the staged split/feedback unitaries, and decomposes each relevant unitary into rank-one phase factors. Each rank-one factor gives a Dicke-space target state. A GPG pulse sequence prepares that target from a fixed reference Dicke state; the prepared target then implements the required rank-one phase.

In [ ]:
reference_weight = num_qubits
reference_state = qt.basis(system_dim, reference_weight)

lv_data = gpgs.build_lv_recovery_data(
    recovery_ops,
    reference_weight=reference_weight,
)

target_rows = []
for label, specs in lv_data["spec_groups"].items():
    for spec in specs:
        target_rows.append(
            {
                "factor": label,
                "eig_index": spec["eig_index"],
                "phase_delta": spec["delta_l"],
            }
        )

targets = pd.DataFrame(target_rows)
targets

Before replacing the exact recovery factors with GPG pulses, we can run the coherent staged recovery using the exact split and feedback unitaries produced above.

In [ ]:
stage_ops_exact = gpgs.recovery_stage_ops(
    lv_data["U_split_exact"],
    lv_data["feedback_ops"],
    system_dim=system_dim,
    prefix="exact",
)
_, rho_exact_circuit, stage_trace = gpgs.run_three_ancilla_recovery_with_ops(
    rho_noisy,
    stage_ops_exact,
    return_records=True,
)
exact_circuit_fidelity = gpgs.output_fidelity(
    rho_exact_circuit,
    rho_code,
    logical_kets=(ket0, ket1),
)

pd.DataFrame(stage_trace), exact_circuit_fidelity

## Optimize pulses for an arbitrary Dicke-space target

This is the core state-preparation primitive. Give `optimize_state_gpg` any normalized ket in the Dicke top block, choose a reference state, and it returns a five-parameter detuned pulse table with columns `alpha`, `beta`, `gamma`, `kappa`, and `detuning`.

In [ ]:
PULSE_COLUMNS = ["alpha", "beta", "gamma", "kappa", "detuning"]


def pulse_dataframe(sequence):
    return pd.DataFrame(
        gpgs.coerce_detuned_pulse_params(sequence),
        columns=PULSE_COLUMNS,
    )


rng = np.random.default_rng(12)
target_vec = rng.normal(size=system_dim) + 1j * rng.normal(size=system_dim)
target_vec = target_vec / np.linalg.norm(target_vec)
target_state = qt.Qobj(target_vec.reshape(-1, 1), dims=[[system_dim], [1]])

state_prep_result = gpgs.optimize_state_gpg(
    num_qubits,
    target_state,
    pulses=4,
    initial_state=reference_state,
    restarts=2,
    maxiter=200,
    seed=12,
    cooperativity=cooperativity,
)

print(f"Arbitrary target infidelity = {state_prep_result.fbest:.3e}")
pulse_dataframe(state_prep_result.X_with_detuning)

## Optimize one recovery target

A recovery synthesis target has the same form as the arbitrary state above, but it comes from a spectral decomposition of one split or feedback unitary. The result is one pulse table keyed by `(factor label, eigenvector index)`.

In [ ]:
first_label = next(label for label, specs in lv_data["spec_groups"].items() if specs)
first_spec = lv_data["spec_groups"][first_label][0]

single_target_settings = {
    "pulses": 5,
    "restarts": 2,
    "maxiter": 250,
    "seed": 2024,
}
single_target_record = gpgs.optimize_recovery_target(
    num_qubits,
    first_spec["target"],
    single_target_settings,
    reference_state,
    first_label,
    int(first_spec["eig_index"]),
    cooperativity=cooperativity,
)

print(first_label, first_spec["eig_index"])
print(f"Recovery-target infidelity = {single_target_record['1 - F_state']:.3e}")
pulse_dataframe(single_target_record["X_detuned"])

## Full GPG recovery synthesis

The full synthesis repeats the previous optimization for every nontrivial rank-one phase target in the split and feedback unitaries, then assembles those pulse sequences into the staged recovery circuit. The teaching settings below are deliberately small; for production-quality sequences, increase the pulse count, restarts, and `maxiter`, or use the defaults in `gpgs.default_gpg_recovery_settings`.

In [ ]:
def teaching_settings(label, eig_index, counter):
    settings = gpgs.default_gpg_recovery_settings(label, eig_index, counter).copy()
    settings["pulses"] = min(settings["pulses"], 5)
    settings["restarts"] = 1
    settings["maxiter"] = 150
    return settings


RUN_FULL_RECOVERY_SYNTHESIS = False

if RUN_FULL_RECOVERY_SYNTHESIS:
    lv_data_full = gpgs.build_lv_recovery_data(
        recovery_ops,
        reference_weight=reference_weight,
    )
    synthesis = gpgs.synthesize_gpg_recovery_from_lv_data(
        lv_data_full,
        settings_fn=teaching_settings,
        cooperativity=cooperativity,
        log=print,
    )
    _, rho_gpg = gpgs.run_three_ancilla_recovery_with_ops(
        rho_noisy,
        synthesis["stage_ops_gpg"],
    )
    gpg_fidelity = gpgs.output_fidelity(
        rho_gpg,
        rho_code,
        logical_kets=(ket0, ket1),
    )
    print(f"GPG recovered codespace fidelity = {gpg_fidelity:.12f}")
    display(pd.DataFrame(synthesis["optimization_records"]))
    display(pd.DataFrame(synthesis["synthesis_quality"]))
else:
    print("Set RUN_FULL_RECOVERY_SYNTHESIS = True to optimize every recovery target.")

The synthesized pulse sequences are available as `synthesis["pulse_sequences"]`, keyed by `(factor label, eig_index)`. Each value is a detuned pulse table with columns `alpha`, `beta`, `gamma`, `kappa`, and `detuning`.